In [1]:
print("Test")

Test


In [2]:
import pandas as pd
import evaluate

In [3]:
ud_test_df = pd.read_csv("../data/ud_et_edt/processed/UD_test.csv")

In [6]:
print(ud_test_df.head())

   sentence_id       words  form pos   file_prefix                   source  \
0            0       Palju   NaN   D  aja_ee199920  et_edt-ud-test_009.json   
1            0     olulisi  pl p   A  aja_ee199920  et_edt-ud-test_009.json   
2            0  komponente  pl p   S  aja_ee199920  et_edt-ud-test_009.json   
3            0           ,   NaN   Z  aja_ee199920  et_edt-ud-test_009.json   
4            0        nagu   NaN   J  aja_ee199920  et_edt-ud-test_009.json   

   labels  
0       D  
1  pl p_A  
2  pl p_S  
3       Z  
4       J  


In [ ]:
poseval = evaluate.load("evaluate-metric/poseval", module_type="metric")


def custom_metrics(preds, labels):

    # Evaluate using poseval
    result = poseval.compute(predictions=preds, references=labels)

    return result

In [ ]:
# 1) Load libs + helpers
import pandas as pd
import torch
from transformers import DataCollatorForTokenClassification
from seqeval.metrics import classification_report
import sys
import pathlib

# Add the project's root directory to the Python path
sys.path.append(pathlib.Path("../").resolve().as_posix())

# import helper functions from your preprocessing module
from scripts.model.preprocessing import (
    initialize_model,
    _group_sentences_from_df,
    prepare_token_classification_data,
    TokenClassificationDataset,
)

In [9]:
# 2) Read test data and keep required cols
test_df = pd.read_csv("../data/ud_et_edt/processed/UD_test.csv")
test_df = test_df[["sentence_id", "words", "labels"]]

In [10]:
# 3) Load model+tokenizer (pass model folder or checkpoint)
model_bundle = initialize_model(
    "../models/NER_mudel_v2_homonym/"
)  # returns model, tokenizer, id2label, label2id
model = model_bundle["model"]
tokenizer = model_bundle["tokenizer"]
id2label = model_bundle["id2label"]
label2id = model_bundle["label2id"]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [11]:
# 4) Group into sentence-level (list of (words_list, labels_list))
sentences = _group_sentences_from_df(test_df)

In [12]:
# 5) Build encodings & aligned labels (use same max_length you trained with)
encodings, aligned_labels = prepare_token_classification_data(
    tokenizer,
    sentences,
    label2id,
    max_length=None,  # adjust if you used different length
    only_target_token=False,  # choose True if you want to ignore non-target placeholders
)

In [13]:
# 6) Create dataset + dataloader
dataset = TokenClassificationDataset(encodings, aligned_labels)
data_collator = DataCollatorForTokenClassification(tokenizer)
dataloader = torch.utils.data.DataLoader(
    dataset, batch_size=8, collate_fn=data_collator
)

In [14]:
# 7) Eval loop: collect token-level predictions and gold labels (ignore -100)
model.eval()
preds_all = []
labels_all = []
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
with torch.no_grad():
    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits.detach().cpu().numpy()
        label_ids = batch["labels"].detach().cpu().numpy()
        pred_ids = logits.argmax(axis=-1)
        for p_row, l_row in zip(pred_ids, label_ids):
            preds = []
            labs = []
            for p, l in zip(p_row, l_row):
                if l == -100:
                    continue
                preds.append(id2label[int(p)])
                labs.append(id2label[int(l)])
            preds_all.append(preds)
            labels_all.append(labs)

In [19]:
# 8) Metrics
results = custom_metrics(preds_all, labels_all)
print(classification_report(labels_all, preds_all))

e:\Git_projects\EstNLTK\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\Git_projects\EstNLTK\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\Git_projects\EstNLTK\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\Git_projec

              precision    recall  f1-score   support

           _       0.50      0.21      0.29     10543
          _V       0.97      0.41      0.58      3170
         a_V       1.00      0.03      0.05      1160
        ad_V       0.91      0.67      0.77       464
       aks_V       0.00      0.00      0.00        15
      akse_V       1.00      0.10      0.18       141
       ama_V       0.00      0.00      0.00         2
        as_V       1.00      0.07      0.14        27
       ast_V       0.00      0.00      0.00         3
        at_V       1.00      0.10      0.18        10
       ata_V       1.00      0.11      0.19        28
        dt_A       1.00      0.11      0.20         9
        dt_H       0.07      0.20      0.11        15
        dt_N       0.00      0.00      0.00         3
        dt_O       0.00      0.00      0.00         2
        dt_P       0.00      0.00      0.00         2
        dt_S       0.20      0.25      0.22       202
        dt_Y       0.00    

In [33]:
# Print weighted average metrics as percentages
print(f"Accuracy: \t{results['accuracy']:.2%}")
print(f"Precision: \t{results['weighted avg']['precision']:.2%}")
print(f"Recall: \t{results['weighted avg']['recall']:.2%}")
print(f"F1-score: \t{results['weighted avg']['f1-score']:.2%}")

Accuracy: 	42.43%
Precision: 	72.46%
Recall: 	42.43%
F1-score: 	41.16%
